In [19]:
import pandas as pd
import numpy as np

print("Jupyter Notebook is ready!")
print("Pandas version:", pd.__version__)

Jupyter Notebook is ready!
Pandas version: 3.0.6


In [21]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

DATA_DIR = Path("data")
REPORT_DIR = Path("reports")
CONFIG_DIR = Path("config")
EVIDENCE_DIR = Path("evidence")

for folder in [DATA_DIR, REPORT_DIR, CONFIG_DIR, EVIDENCE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RAW_FILE = DATA_DIR / "raw_dataset.csv"
CLEAN_FILE = DATA_DIR / "cleaned_dataset.xlsx"
CHANGE_LOG_FILE = REPORT_DIR / "change_log.csv"
QUALITY_REPORT_FILE = REPORT_DIR / "quality_report.txt"

print("Folders ready.")
print("Raw file expected at:", RAW_FILE.resolve())


Folders ready.
Raw file expected at: C:\Users\dell\Documents\Veda Technologies\data-cleaning-preprocessing\data\raw_dataset.csv


In [ ]:
if not RAW_FILE.exists():
    raise FileNotFoundError(
        f"Cannot find {RAW_FILE}. Download a Titanic CSV and save it as "
        f"'{RAW_FILE.as_posix()}', then run this cell again."
    )

df_raw = pd.read_csv(RAW_FILE)
df = df_raw.copy(deep=True)

print("Raw dataset loaded successfully.")
print("Rows:", df_raw.shape[0])
print("Columns:", df_raw.shape[1])
display(df_raw.head())


Raw dataset loaded successfully.
Rows: 418
Columns: 12


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [ ]:
print("Dataset information:")
df_raw.info()

print("\nMissing values by column:")
display(df_raw.isna().sum().rename("missing_count").to_frame())

print("\nExact duplicate rows:", int(df_raw.duplicated().sum()))

print("\nColumn data types:")
display(df_raw.dtypes.astype(str).rename("data_type").to_frame())

print("\nSummary statistics:")
display(df_raw.describe(include="all").T)


Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Survived     418 non-null    int64  
 2   Pclass       418 non-null    int64  
 3   Name         418 non-null    str    
 4   Sex          418 non-null    str    
 5   Age          332 non-null    float64
 6   SibSp        418 non-null    int64  
 7   Parch        418 non-null    int64  
 8   Ticket       418 non-null    str    
 9   Fare         417 non-null    float64
 10  Cabin        91 non-null     str    
 11  Embarked     418 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 39.3 KB

Missing values by column:


,missing_count
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,86
SibSp,0
Parch,0
Ticket,0
Fare,1



Exact duplicate rows: 0

Column data types:


,data_type
PassengerId,int64
Survived,int64
Pclass,int64
Name,str
Sex,str
Age,float64
SibSp,int64
Parch,int64
Ticket,str
Fare,float64



Summary statistics:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
PassengerId,418.0,NaN,NaN,NaN,1100.5,120.810458,892.0,996.25,1100.5,1204.75,1309.0
Survived,418.0,NaN,NaN,NaN,0.363636,0.481622,0.0,0.0,0.0,1.0,1.0
Pclass,418.0,NaN,NaN,NaN,2.26555,0.841838,1.0,1.0,3.0,3.0,3.0
Name,418,418,"Kelly, Mr. James",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sex,418,2,male,266,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,332.0,NaN,NaN,NaN,30.27259,14.181209,0.17,21.0,27.0,39.0,76.0
SibSp,418.0,NaN,NaN,NaN,0.447368,0.89676,0.0,0.0,0.0,1.0,8.0
Parch,418.0,NaN,NaN,NaN,0.392344,0.981429,0.0,0.0,0.0,0.0,9.0
Ticket,418,363,PC 17608,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Fare,417.0,NaN,NaN,NaN,35.627188,55.907576,0.0,7.8958,14.4542,31.5,512.3292


In [ ]:
initial_rows = len(df)
initial_columns = len(df.columns)
missing_before = int(df.isna().sum().sum())
duplicates_before = int(df.duplicated().sum())

change_log = []

def log_change(issue, action, affected_count, details):
    change_log.append({
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "issue": issue,
        "action": action,
        "affected_count": int(affected_count),
        "details": str(details)
    })

log_change("Missing values", "Inventory", missing_before,
           "Total missing cells before cleaning")
log_change("Duplicate rows", "Inventory", duplicates_before,
           "Exact duplicate rows before cleaning")

print({
    "rows_before": initial_rows,
    "columns_before": initial_columns,
    "missing_cells_before": missing_before,
    "duplicate_rows_before": duplicates_before
})


{'rows_before': 418, 'columns_before': 12, 'missing_cells_before': 414, 'duplicate_rows_before': 0}


In [ ]:
old_columns = list(df.columns)
df.columns = [str(col).strip() for col in df.columns]
log_change("Column names", "Trimmed whitespace", 
           sum(str(old) != str(new) for old, new in zip(old_columns, df.columns)),
           "Removed leading/trailing whitespace from column names")

text_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
for col in text_cols:
    df[col] = df[col].map(
        lambda value: value.strip() if isinstance(value, str) else value
    )

blank_markers = ["", "NA", "N/A", "n/a", "null", "NULL", "None", "?"]
blank_count = 0
for col in text_cols:
    before_nonnull = int(df[col].notna().sum())
    df[col] = df[col].replace(blank_markers, np.nan)
    blank_count += before_nonnull - int(df[col].notna().sum())

log_change("Text formatting", "Trimmed whitespace and normalized blank markers",
           blank_count, "Common text placeholders were converted to missing values")
print("Text formatting complete.")


Text formatting complete.


In [ ]:
rows_before_dedup = len(df)
df = df.drop_duplicates().copy()
duplicates_removed = rows_before_dedup - len(df)

log_change("Duplicate rows", "Removed exact duplicate rows",
           duplicates_removed, "Only rows identical across all columns were removed")
print("Duplicate rows removed:", duplicates_removed)
print("Rows remaining:", len(df))


Duplicate rows removed: 0
Rows remaining: 418


In [ ]:
possible_numeric_cols = [
    "PassengerId", "Survived", "Pclass", "Age", "SibSp", "Parch", "Fare"
]
conversion_results = []

for col in possible_numeric_cols:
    if col in df.columns:
        non_missing_before = df[col].notna()
        converted = pd.to_numeric(df[col], errors="coerce")
        failed = int((non_missing_before & converted.isna()).sum())
        df[col] = converted
        conversion_results.append({
            "column": col,
            "conversion_failures": failed,
            "result_dtype": str(df[col].dtype)
        })
        log_change(f"Data type: {col}", "Converted to numeric",
                   failed, "Unparseable non-missing values became missing and require review")

display(pd.DataFrame(conversion_results))


,column,conversion_failures,result_dtype
0,PassengerId,0,int64
1,Survived,0,int64
2,Pclass,0,int64
3,Age,0,float64
4,SibSp,0,int64
5,Parch,0,int64
6,Fare,0,float64


In [ ]:
category_cols = [col for col in ["Sex", "Embarked"] if col in df.columns]
for col in category_cols:
    before_values = df[col].copy()
    df[col] = df[col].map(
        lambda value: value.title() if isinstance(value, str) else value
    )
    changed = int((before_values.fillna("__MISSING__") !=
                   df[col].fillna("__MISSING__")).sum())
    log_change(f"Category formatting: {col}", "Standardized capitalization",
               changed, "Applied title case to selected category values")

print("Standardized columns:", category_cols)


Standardized columns: ['Sex', 'Embarked']


In [ ]:
imputation_results = []

for col in df.columns:
    missing_count = int(df[col].isna().sum())
    if missing_count == 0:
        continue

    if col in ["Age", "Fare"] and pd.api.types.is_numeric_dtype(df[col]):
        fill_value = df[col].median()
        if pd.notna(fill_value):
            df[col] = df[col].fillna(fill_value)
            log_change(f"Missing values: {col}", "Median imputation",
                       missing_count, f"Filled with median value {fill_value}")
            imputation_results.append({
                "column": col, "method": "median",
                "values_filled": missing_count, "fill_value": fill_value
            })
        else:
            imputation_results.append({
                "column": col, "method": "retained",
                "values_filled": 0, "fill_value": None
            })

    elif col == "Cabin":
        # Cabin is often missing for many passengers; do not invent cabin numbers.
        log_change(f"Missing values: {col}", "Retained missing values",
                   missing_count, "Cabin values were not invented")
        imputation_results.append({
            "column": col, "method": "retained_missing",
            "values_filled": 0, "fill_value": None
        })

    elif col in ["Sex", "Embarked"] or (
        df[col].dtype == "object" and df[col].nunique(dropna=True) <= 10
    ):
        modes = df[col].mode(dropna=True)
        if not modes.empty:
            fill_value = modes.iloc[0]
            df[col] = df[col].fillna(fill_value)
            log_change(f"Missing values: {col}", "Mode imputation",
                       missing_count, f"Filled with most frequent value {fill_value}")
            imputation_results.append({
                "column": col, "method": "mode",
                "values_filled": missing_count, "fill_value": fill_value
            })
        else:
            log_change(f"Missing values: {col}", "Retained missing values",
                       missing_count, "No non-missing mode was available")
            imputation_results.append({
                "column": col, "method": "retained_missing",
                "values_filled": 0, "fill_value": None
            })

    else:
        log_change(f"Missing values: {col}", "Retained for review",
                   missing_count, "No automatic imputation applied")
        imputation_results.append({
            "column": col, "method": "retained_for_review",
            "values_filled": 0, "fill_value": None
        })

display(pd.DataFrame(imputation_results))


,column,method,values_filled,fill_value
0,Age,median,86,27.0000
1,Fare,median,1,14.4542
2,Cabin,retained_missing,0,NaN


In [ ]:
missing_after = int(df.isna().sum().sum())
duplicates_after = int(df.duplicated().sum())

quality_by_column = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "missing_values": df.isna().sum(),
    "unique_values": df.nunique(dropna=True)
})

print("FINAL QUALITY CHECK")
print("===================")
print("Rows before:", initial_rows)
print("Rows after:", len(df))
print("Columns before:", initial_columns)
print("Columns after:", len(df.columns))
print("Missing cells before:", missing_before)
print("Missing cells after:", missing_after)
print("Duplicate rows before:", duplicates_before)
print("Duplicate rows after:", duplicates_after)

display(quality_by_column)

assert duplicates_after == 0, "Exact duplicate rows remain; investigate before submission."
print("Duplicate-row validation passed.")


FINAL QUALITY CHECK
Rows before: 418
Rows after: 418
Columns before: 12
Columns after: 12
Missing cells before: 414
Missing cells after: 327
Duplicate rows before: 0
Duplicate rows after: 0


,data_type,missing_values,unique_values
PassengerId,int64,0,418
Survived,int64,0,2
Pclass,int64,0,3
Name,str,0,418
Sex,str,0,2
Age,float64,0,79
SibSp,int64,0,7
Parch,int64,0,8
Ticket,str,0,363
Fare,float64,0,169


Duplicate-row validation passed.


In [ ]:
df.to_excel(CLEAN_FILE, index=False)
pd.DataFrame(change_log).to_csv(CHANGE_LOG_FILE, index=False)

print("Saved cleaned dataset:", CLEAN_FILE.resolve())
print("Saved change log:", CHANGE_LOG_FILE.resolve())


Saved cleaned dataset: C:\Users\dell\Documents\Veda Technologies\data-cleaning-preprocessing\data\cleaned_dataset.xlsx
Saved change log: C:\Users\dell\Documents\Veda Technologies\data-cleaning-preprocessing\reports\change_log.csv


In [ ]:
report_lines = [
    "DATA CLEANING AND PREPROCESSING — QUALITY REPORT",
    "=" * 52,
    f"Generated: {datetime.now().isoformat(timespec='seconds')}",
    f"Input file: {RAW_FILE.as_posix()}",
    f"Cleaned output: {CLEAN_FILE.as_posix()}",
    "",
    f"Rows before: {initial_rows}",
    f"Rows after: {len(df)}",
    f"Columns before: {initial_columns}",
    f"Columns after: {len(df.columns)}",
    f"Missing cells before: {missing_before}",
    f"Missing cells after: {missing_after}",
    f"Duplicate rows before: {duplicates_before}",
    f"Duplicate rows after: {duplicates_after}",
    "",
    "Remaining missing values by column:",
    df.isna().sum().to_string(),
    "",
    "Final data types:",
    df.dtypes.astype(str).to_string(),
    "",
    "Review reports/change_log.csv for documented cleaning actions.",
    "Review retained missing values and conversion failures before analysis."
]

report_text = "\n".join(report_lines)
QUALITY_REPORT_FILE.write_text(report_text, encoding="utf-8")
print(report_text)
print("\nSaved report:", QUALITY_REPORT_FILE.resolve())


DATA CLEANING AND PREPROCESSING — QUALITY REPORT
Generated: 2026-10-20T18:21:54
Input file: data/raw_dataset.csv
Cleaned output: data/cleaned_dataset.xlsx

Rows before: 418
Rows after: 418
Columns before: 12
Columns after: 12
Missing cells before: 414
Missing cells after: 327
Duplicate rows before: 0
Duplicate rows after: 0

Remaining missing values by column:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          327
Embarked         0

Final data types:
PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str

Review reports/change_log.csv for documented cleaning actions.
Review retained missing values and conversion fail